# 03 SQL 查询与浏览权重敏感性

用 SQL 汇总用户与筛选候选名单，再固定前 200 名比较不同浏览权重。

In [1]:
from pathlib import Path
import sys
# 兼容从项目根目录或 code 目录打开。
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'code/analysis.py').exists())
if str(ROOT / 'code') not in sys.path:
    sys.path.insert(0, str(ROOT / 'code'))
import pandas as pd
from IPython.display import display, Image


In [2]:
from analysis import run_sql, compare_weights
users = pd.read_csv(ROOT/'outputs/user_segments.csv')
recomputed = pd.read_csv(ROOT/'outputs/user_segments_recomputed.csv')

## SQL 查询
采用 SQLite。三组查询分别是分层画像汇总、基础分前 200 名，以及剔除后重算模型中“潜力且已订阅”的实验候选名单。

In [3]:
print((ROOT/'code/analysis.sql').read_text(encoding='utf-8'))

-- SQLite 语法。users 为全量分层结果；users_recomputed 为剔除后重算结果。
-- name: segment_summary
SELECT
    Segment_Name,
    COUNT(*) AS user_count,
    ROUND(AVG(Total_Spending), 2) AS average_spending,
    ROUND(AVG(Purchase_Frequency), 2) AS average_frequency,
    ROUND(AVG(I_Score), 2) AS average_browsing_score
FROM users
GROUP BY Segment_Name
ORDER BY user_count DESC, Segment_Name;

-- name: top_200_rfm
WITH ranked AS (
    SELECT User_ID, Segment_Name, RFM_Score,
           ROW_NUMBER() OVER (ORDER BY RFM_Score DESC, User_ID) AS ranking
    FROM users
)
SELECT * FROM ranked
WHERE ranking <= 200
ORDER BY ranking;

-- name: eligible_experiment_users
SELECT User_ID, Segment_Name, RFM_Score, I_Score, Final_Score
FROM users_recomputed
WHERE Group_Type = '潜力'
  AND Newsletter_Subscription = 1
ORDER BY Final_Score DESC, User_ID;



In [4]:
sql_results = run_sql(users, recomputed)
display(sql_results['segment_summary'])
display(sql_results['top_200_rfm'].head(10))
print('实验候选人数', len(sql_results['eligible_experiment_users']))

,Segment_Name,user_count,average_spending,average_frequency,average_browsing_score
0,一般用户,443,2368.12,5.17,48.80
1,一般发展用户,81,1117.95,6.43,49.84
2,重要挽留用户,67,4134.42,2.57,46.53
3,高收入消费较高低活跃用户,64,3990.70,4.92,44.09
4,低价值用户,50,1036.52,1.68,42.79
5,重要发展用户,48,3873.96,1.63,48.91
6,一般维持用户,47,1055.40,1.47,45.83
7,重要价值用户,47,3960.19,7.47,42.28
8,重要保持用户,39,4030.23,7.49,48.00
9,隐形活跃者,34,2756.79,1.32,77.99


,User_ID,Segment_Name,RFM_Score,ranking
0,#575,重要价值用户,96.770733,1
1,#203,重要价值用户,93.570405,2
2,#450,重要价值用户,92.494081,3
3,#864,重要价值用户,91.984712,4
4,#607,核心VIP,90.138268,5
5,#391,重要价值用户,89.942267,6
6,#919,核心VIP,89.223158,7
7,#434,重要价值用户,89.116022,8
8,#963,重要价值用户,88.115262,9
9,#10,一般用户,88.095238,10


实验候选人数 40


## 控制入选人数，改变浏览权重
权重使用 0%、10%、20%、30%；每次固定选择 200 人。同分时按用户 ID 升序，确保重复运行得到同一份名单。

In [5]:
weight_comparison = compare_weights(users, weights=(0,0.1,0.2,0.3), top_n=200)
weight_comparison.round(4)

,浏览加权幅度,人数,与基础名单重合比例,替换人数,平均累计消费,平均浏览得分
0,0.0,200,1.000,0,4117.730,48.8323
1,0.1,200,0.955,9,4109.840,51.0485
2,0.2,200,0.910,18,4084.860,53.0744
3,0.3,200,0.880,24,4045.815,54.7577


## 读取名单变化
20% 权重下替换 18 人，重合率 91%。入选平均浏览得分上升，但平均历史累计消费略降。这是排序取舍，不是已验证的收入提升。

In [6]:
# 固定选前 200 人，比较加入浏览加权后新进入的用户。
base = users.sort_values(['RFM_Score','User_ID'], ascending=[False,True]).head(200)
weighted = users.sort_values(['Final_Score','User_ID'], ascending=[False,True]).head(200)
new_entries = weighted.loc[~weighted.User_ID.isin(base.User_ID)]
print('新入选人数', len(new_entries))
new_entries[['User_ID','Segment_Name','RFM_Score','I_Score','Final_Score']].head(10)

新入选人数 18


,User_ID,Segment_Name,RFM_Score,I_Score,Final_Score
899,#900,一般用户,66.713438,82.956449,77.782058
0,#1,一般用户,65.378994,87.285385,76.792255
97,#98,高收入消费较高低活跃用户,65.060949,84.333124,76.034535
246,#247,一般用户,65.818498,75.418760,75.746397
795,#796,一般用户,64.331755,87.541876,75.595200
760,#761,一般用户,64.822123,82.291667,75.490764
765,#766,一般用户,64.266713,85.322446,75.233500
125,#126,一般用户,64.256921,84.767588,75.150729
140,#141,一般用户,65.634482,71.042714,74.960186
929,#930,隐形活跃者,63.731328,86.149497,74.712172


查询输出在 `outputs/sql/`；权重汇总和入选名单在 `outputs/weights/`。